# LPSE-X Training Pipeline

Find IT! 2026 â€” Track C Phase 2

This notebook demonstrates the complete training pipeline:
1. Data loading and temporal splitting
2. Feature engineering (30 features)
3. Heuristic labeling (7 red flags)
4. XGBoost HPO with Optuna
5. Model evaluation and calibration
6. SHAP explainability

**Note:** This notebook imports from `src/` modules. HPO is skipped if a trained model already exists.

In [ ]:
import json
import logging
import numpy as np
import pandas as pd
import xgboost as xgb

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from src.data import PROJECT_ROOT, PROCESSED_DIR
from src.split import external_raw_split, internal_dev_splits, TRAIN_DIR, TEST_DIR
from src.features import compute_all_features
from src.labels import compute_risk_labels
from src.model import (
    load_model, load_train_artifacts, load_test_artifacts,
    evaluate, save_metrics, MODELS_DIR, CLASS_NAMES,
    run_calibration, compute_imputation_values,
    export_onnx, run_evaluation_pipeline
)
from src.explain import get_explainer, explain_single, generate_shap_summary

print('All imports OK')

## 1. Load Materialized Data

Data has been pre-processed by `scripts/materialize.py` and stored as Parquet.

In [ ]:
train_features, train_labels = load_train_artifacts()
test_features, test_labels = load_test_artifacts()

print(f'Train: {train_features.shape[0]} rows, {train_features.shape[1]} features')
print(f'Test:  {test_features.shape[0]} rows, {test_features.shape[1]} features')
print(f'\nTrain label distribution:')
print(train_labels.value_counts().sort_index())
print(f'\nTest label distribution:')
print(test_labels.value_counts().sort_index())

## 2. Load Trained Model

The model was trained via `src.model.run_training_pipeline()`. We load the saved `.ubj` model and read `models/metrics.json`.

In [ ]:
model = load_model()
print(f'Model loaded: {model.num_boosted_rounds()} trees')

# Read canonical metrics
metrics_path = MODELS_DIR / 'metrics.json'
metrics = json.loads(metrics_path.read_text())
print(f'\nNote: {metrics["note"]}')

test_m = metrics.get('final_test', {})
print(f'\nTest Accuracy: {test_m.get("accuracy", "N/A")}')
print(f'Test Macro-F1: {test_m.get("macro_f1", "N/A")}')
print(f'Per-class F1: {test_m.get("per_class_f1", "N/A")}')

## 3. Evaluate on Test Set

In [ ]:
test_metrics = evaluate(model, test_features, test_labels, 'test')
print(json.dumps({k: v for k, v in test_metrics.items() if k != 'classification_report'}, indent=2))

## 4. Calibration

Temperature scaling using reviewed clean labels from `val_calibration`.

In [ ]:
calibration = json.loads((MODELS_DIR / 'calibration.json').read_text())
print(f'Calibration enabled: {calibration["enabled"]}')
if calibration['enabled']:
    print(f'Temperature: {calibration["temperature"]:.4f}')
    print(f'Calibration samples: {calibration["n_calibration_samples"]}')

## 5. SHAP Feature Importance

In [ ]:
explainer = get_explainer(model)

# Explain a test sample
sample_row = test_features.iloc[[0]]
explanation = explain_single(sample_row, model=model, explainer=explainer)

print(f'Predicted: {explanation["predicted_label"]} ({explanation["probability"]:.1%})')
print(f'\nTop contributing factors:')
for f in explanation['factors']:
    print(f'  {f["feature"]}: SHAP={f["shap_value"]:.4f} ({f["direction"]})')

## 6. Generate SHAP Summary Plot

In [ ]:
shap_path = generate_shap_summary(model, test_features)
print(f'SHAP summary saved to: {shap_path}')

from IPython.display import Image, display
display(Image(filename=str(shap_path)))

## 7. Export ONNX and Imputation Values

In [ ]:
imputation = compute_imputation_values(train_features)
onnx_path = export_onnx(model, train_features)
print(f'ONNX model: {onnx_path}')
print(f'Imputation values: {len(imputation)} features')

## Summary

Training pipeline complete. Artifacts saved:
- `models/xgb_model.ubj` â€” trained model
- `models/best_params.json` â€” HPO best parameters
- `models/metrics.json` â€” canonical evaluation metrics
- `models/calibration.json` â€” temperature scaling config
- `models/imputation_values.json` â€” median imputation values
- `proposal/figures/shap_summary.png` â€” SHAP feature importance